In [5]:
import pandas as pd
import pyodbc
from datetime import timedelta
import os

def generate_llm_friendly_report(db_path, target_fund_code, start_date, end_date, output_md_path, perf_df, sdi_df):


    # --- 指标深度解释手册 (Handbook) ---
    METRIC_HANDBOOK = {
        "TMTiming (TM择时能力)": (
            "定义：基于 Treynor-Mazuy 模型，衡量经理通过调整仓位或组合 Beta 捕捉市场波动的能力。\n"
            "评价：百分位排名越小（如 <10%）择时能力越强。优秀者通常能有效进行‘高抛低吸’。"
        ),
        "TMStkSlct (TM选股能力)": (
            "定义：衡量经理剔除市场波动后，挑选个股获得 Alpha 超额收益的能力。\n"
            "评价：核心主动管理指标。排名越前，代表收益更多源于深度研报和个股挖掘，而非跟随大盘。"
        ),
        "SDI (选股驱动力)": (
            "定义：Selection Driving Index。衡量收益中由‘个股选择’驱动的比例，而非风格或行业暴露。\n"
            "评价：数值越高，代表投资风格越‘纯粹’，不依赖市场博弈或频繁的行业轮动。"
        )
    }

    def convert_to_percent(rank_str):
        if pd.isna(rank_str) or not isinstance(rank_str, str) or '/' not in rank_str:
            return "-"
        try:
            parts = rank_str.split('/')
            rank, total = float(parts[0]), float(parts[1])
            if total == 0: return "-"
            return f"{round((rank / total) * 100, 1)}%"
        except:
            return "-"

    # --- 数据库连接 ---
    conn_str = (
        r'DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};'
        rf'DBQ={db_path};'
    )

    try:
        conn = pyodbc.connect(conn_str)
        df_all = pd.read_sql("SELECT * FROM [FundManager]", conn)
        conn.close()
        df_all.columns = df_all.columns.str.strip()
    except Exception as e:
        print(f"数据库读取失败: {e}")
        return

    # 数据标准化处理
    df_all['MasterFundCode'] = df_all['MasterFundCode'].astype(str).str.split('.').str[0].str.zfill(6)
    target_fund_code_clean = str(target_fund_code).split('.')[0].zfill(6)
    df_all['ServiceStartDate_DT'] = pd.to_datetime(df_all['ServiceStartDate'], errors='coerce')
    df_all['ServiceEndDate_DT'] = pd.to_datetime(df_all['ServiceEndDate'], errors='coerce')

    t_start, t_end = pd.to_datetime(start_date), pd.to_datetime(end_date)

    # 定位目标基金经理
    mask = (df_all['MasterFundCode'] == target_fund_code_clean) & \
           (df_all['ServiceStartDate_DT'] <= t_end) & \
           (df_all['ServiceEndDate_DT'].fillna(pd.Timestamp('2099-12-31')) >= t_start)

    target_ids = df_all[mask]['FundManagerID'].unique()

    if len(target_ids) == 0:
        print(f"未找到基金 {target_fund_code_clean} 的经理记录。")
        return

    # --- 开始生成 Markdown 报告 ---
    with open(output_md_path, 'w', encoding='utf-8') as f:
        f.write(f"# FUND_ANALYSIS_REPORT: {target_fund_code_clean}\n\n")

        # 写入深度指标手册（增强 LLM 的解释能力）
        f.write("### [METRIC_DEFINITION_AND_GUIDE]\n")
        f.write("> **Note**: All values are **Percentile Ranks** (0.0% to 100.0%). Lower values represent higher performance/rank.\n\n")
        for title, desc in METRIC_HANDBOOK.items():
            f.write(f"#### {title}\n{desc}\n\n")
        f.write("---\n\n")

        for m_id in target_ids:
            m_records = df_all[df_all['FundManagerID'] == m_id]
            m_info = m_records.sort_values('ServiceStartDate', ascending=False).iloc[0]

            f.write(f"## === MANAGER: {m_info['ManagerName']} ({m_id}) ===\n")
            f.write(f"### [PROFILE]\n- **Degree**: {m_info.get('Degree', 'N/A')} | **Univ**: {m_info.get('University', 'N/A')} | **Major**: {m_info.get('Major', 'N/A')}\n")
            f.write(f"### [RESUME]\n{m_info.get('Resume', 'N/A')[:500]}...\n\n")

            f.write("### [OTHER_MANAGED_FUNDS_ANALYSIS]\n")
            other_managed = m_records[m_records['MasterFundCode'] != target_fund_code_clean]

            if other_managed.empty:
                f.write("无其他历史管理记录。\n")
            else:
                for _, row in other_managed.sort_values('ServiceStartDate', ascending=False).iterrows():
                    f_code = row['MasterFundCode']
                    f_name = row.get('FundName', 'Unknown')
                    f.write(f"#### Fund: {f_code} ({f_name})\n")

                    # --- 整合 SDI / 类别 / 风格信息 ---
                    if not sdi_df.empty:
                        fund_extra = sdi_df[sdi_df['MasterFundCode'] == f_code]
                        if not fund_extra.empty:
                            sdi_val = fund_extra.iloc[0].get('SDI', '-')
                            cat = fund_extra.iloc[0].get('Category', '-')
                            style = fund_extra.iloc[0].get('InvestmentStyle', '-')
                            f.write(f"- **Attributes**: Category: {cat} | Style: {style} | SDI: {sdi_val}\n")
                        else:
                            f.write("- **Attributes**: No SDI or style metadata available.\n")

                    # --- 业绩排名百分位表 ---
                    sub_perf = perf_df[perf_df['Symbol'] == f_code].copy()
                    if not sub_perf.empty:
                        sub_perf['TradingDate_DT'] = pd.to_datetime(sub_perf['TradingDate'])
                        latest_date = sub_perf['TradingDate_DT'].max()
                        recent_year = sub_perf[sub_perf['TradingDate_DT'] >= (latest_date - timedelta(days=365))]
                        recent_year = recent_year.sort_values('TradingDate', ascending=False)

                        if not recent_year.empty:
                            rank_cols = [c for c in recent_year.columns if '_FullRnk' in c or '_Rnk' in c]
                            if rank_cols:
                                display_headers = [c.replace('_FullRnk', '').replace('_Rnk', '') for c in rank_cols]
                                f.write("| Date | " + " | ".join(display_headers) + " |\n")
                                f.write("| :--- | " + " | ".join([":---" for _ in display_headers]) + " |\n")

                                for _, p_row in recent_year.iterrows():
                                    p_vals = [convert_to_percent(p_row[c]) for c in rank_cols]
                                    f.write(f"| {p_row['TradingDate']} | {' | '.join(p_vals)} |\n")
                            else:
                                f.write("- 无可用排名数据。\n")
                        else:
                            f.write("- 近12个月无业绩记录。\n")
                    f.write("\n")

            f.write(f"## === END_OF_MANAGER_SECTION ===\n\n")

# ==========================================
# 自动化执行流
# ==========================================
if __name__ == "__main__":
    # 1. 路径配置
    ACCESS_DB = r"C:\Users\qyw\Desktop\智能体\各智能体\基金经理分析智能体\数据\处理后\Fundmanagerinfo.accdb"
    PERFORMANCE_CSV = r"D:\数据\业绩\处理后\FundPerform.csv"
    SDI_EXCEL = r"C:\Users\qyw\Desktop\智能体\各智能体\基金经理分析智能体\数据\处理后\SDI\Fund_SDI_Results.xlsx"

    # 2. 加载业绩大表
    print("正在加载业绩数据库...")
    all_perf_df = pd.read_csv(PERFORMANCE_CSV, encoding='utf-8-sig', low_memory=False)
    all_perf_df['Symbol'] = all_perf_df['Symbol'].astype(str).str.replace(r'\.0$', '', regex=True).str.zfill(6)

    # 3. 加载 SDI 及风格数据 (Excel)
    print("正在从 Excel 加载 SDI 及投资风格数据...")
    sdi_df = pd.DataFrame()
    if os.path.exists(SDI_EXCEL):
        try:
            sdi_df = pd.read_excel(SDI_EXCEL)
            sdi_df['MasterFundCode'] = sdi_df['MasterFundCode'].astype(str).str.split('.').str[0].str.zfill(6)
            print(f"SDI 数据载入成功: {len(sdi_df)} 条记录。")
        except Exception as e:
            print(f"Excel 读取失败，请检查文件是否被占用: {e}")
    else:
        print(f"错误：未找到文件 {SDI_EXCEL}")

    # 4. 执行报告生成
    TARGET_FUNDS = ["000005"]
    START_DATE = "2023-01-01"
    END_DATE = "2026-12-31"

    for code in TARGET_FUNDS:
        output_name = f"{code}_Analysis_Report.md"
        print(f"正在生成基金 {code} 的经理能力穿透分析...")
        generate_llm_friendly_report(
            db_path=ACCESS_DB,
            target_fund_code=code,
            start_date=START_DATE,
            end_date=END_DATE,
            output_md_path=output_name,
            perf_df=all_perf_df,
            sdi_df=sdi_df
        )

    print("\n[任务完成] 报告已保存至当前脚本目录。")

正在加载业绩数据库...
正在从 Excel 加载 SDI 及投资风格数据...
SDI 数据载入成功: 4486 条记录。
正在生成基金 000005 的经理能力穿透分析...


C:\Users\qyw\AppData\Local\Temp\ipykernel_34284\1472867923.py:51: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_all = pd.read_sql("SELECT * FROM [FundManager]", conn)



[任务完成] 报告已保存至当前脚本目录。
